In [0]:
!pip install torch transformers torchvision

In [0]:
%restart_python

Feature Extraction with Parallel Processing

In [0]:
import os
import os.path as osp
import torch
import json
import time
from concurrent.futures import ProcessPoolExecutor, as_completed

# Global variables to be set per worker.
global_feature_extractor = None
global_model = None

def init_worker():
    """Initializer for each worker process: load DinoV2-large model and feature extractor."""
    global global_feature_extractor, global_model

    from transformers import AutoImageProcessor, AutoModel
    global_feature_extractor = AutoImageProcessor.from_pretrained("facebook/dinov2-large", use_fast=True)
    global_model = AutoModel.from_pretrained("facebook/dinov2-large").to("cpu")
    global_model.eval()
    print("Worker initialized with DinoV2-large model.")

def process_frame_embedding(task):
    """
    Processes one cropped frame:
      - Loads the image (assumed to be 224x224 RGB).
      - Applies the DinoV2 feature extractor (which includes ImageNet normalization).
      - Passes the image through the DinoV2 model.
      - Computes the mean-pooled embedding from the last hidden state.
      - Saves the embedding as a .pt file.
      
    Task is a dict with keys:
      - "frame_path": full path to the cropped frame (with /dbfs/ prefix)
      - "object_name": string (e.g., "2633")
      - "output_folder": folder where to save the embedding (with /dbfs/ prefix)
      - "target_size": tuple, expected to be (224, 224)
    """
    try:
        from PIL import Image
        # Open the image (should already be 224x224)
        img = Image.open(task["frame_path"]).convert("RGB")
        # Preprocess the image
        inputs = global_feature_extractor(images=img, return_tensors="pt")
        # Forward pass through the model
        outputs = global_model(**inputs)
        # Get the last hidden state. Assume shape is (1, H, W, D) or (1, H*W, D)
        hidden = outputs.last_hidden_state
        if len(hidden.shape) == 4:
            hidden = hidden.view(hidden.size(0), -1, hidden.size(-1))
        # Mean pooling over spatial dimensions.
        embedding = hidden.mean(dim=1).squeeze().detach().cpu()
        # Build output filename: "<frame_basename>.pt"
        frame_base = osp.splitext(osp.basename(task["frame_path"]))[0]
        out_filename = f"{frame_base}.pt"
        out_path = osp.join(task["output_folder"], out_filename)
        torch.save(embedding, out_path)
        return out_path
    except Exception as e:
        print(f"Error processing frame {task['frame_path']}: {e}")
        return None

def extract_embeddings_for_object(cropped_frames_folder, object_name, output_embeddings_folder, max_workers=8):
    """
    Processes all cropped frames for one object to extract embeddings in parallel.
    
    Parameters:
      - cropped_frames_folder (str): Directory containing cropped frames for the object.
      - object_name (str): Name of the object (e.g., "2633")
      - output_embeddings_folder (str): Base directory where embeddings will be saved.
      - max_workers (int): Number of parallel processes to use.
      
    Returns:
      A dictionary mapping frame base name (without extension) to the saved embedding file path.
    """
    # threshold for maximum frame index to process
    MAX_INDEX = 96000

    # Ensure paths have /dbfs/ prefix.
    if not cropped_frames_folder.startswith("/dbfs"):
        cropped_frames_folder = osp.join("/dbfs", cropped_frames_folder.lstrip("/"))
    if not output_embeddings_folder.startswith("/dbfs"):
        output_embeddings_folder = osp.join("/dbfs", output_embeddings_folder.lstrip("/"))
    
    # Create output folder for this object.
    object_output_folder = osp.join(output_embeddings_folder, object_name)
    os.makedirs(object_output_folder, exist_ok=True)
    
    # List and filter all cropped frame files in the folder.
    valid_exts = (".jpg", ".jpeg", ".png")
    frame_files = []
    for fname in sorted(os.listdir(cropped_frames_folder)):
        if not fname.lower().endswith(valid_exts):
            continue
        # parse leading numeric index before the first underscore
        try:
            idx = int(fname.split('_', 1)[0])
        except ValueError:
            # skip any files not matching the "<number>_..." pattern
            continue
        if idx > MAX_INDEX:
            # skip beyond 96000
            continue
        frame_files.append(osp.join(cropped_frames_folder, fname))

    print(f"Found {len(frame_files)} cropped frames for object '{object_name}' (index ≤ {MAX_INDEX}) in {cropped_frames_folder}.")
    
    tasks = []
    for frame_path in frame_files:
        tasks.append({
            "frame_path": frame_path,
            "object_name": object_name,
            "output_folder": object_output_folder,
            "target_size": (224,224)
        })
    
    results = {}
    with ProcessPoolExecutor(max_workers=max_workers, initializer=init_worker) as executor:
        future_to_task = {executor.submit(process_frame_embedding, task): task for task in tasks}
        for future in as_completed(future_to_task):
            res = future.result()
            if res is not None:
                key = osp.splitext(osp.basename(res))[0]
                results[key] = res
    return results


# NEW: Process multiple object folders one by one.

def extract_embeddings_for_all_objects(input_cropped_frames_parent, output_embeddings_folder, max_workers=8):
    """
    Processes each subfolder (object folder) inside the input cropped frames parent folder.
    """
    if not input_cropped_frames_parent.startswith("/dbfs"):
        input_cropped_frames_parent = osp.join("/dbfs", input_cropped_frames_parent.lstrip("/"))
    
    object_folders = [osp.join(input_cropped_frames_parent, d)
                      for d in os.listdir(input_cropped_frames_parent)
                      if osp.isdir(osp.join(input_cropped_frames_parent, d))]
    all_results = {}
    for obj_folder in object_folders:
        object_name = osp.basename(obj_folder)
        print(f"Processing object folder: {object_name}")
        result = extract_embeddings_for_object(
            obj_folder, object_name, output_embeddings_folder, max_workers=max_workers
        )
        all_results[object_name] = result
    return all_results


# Main

if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_cropped_frames_parent",  
                        default="/xxx/Eem/Cropped Frames/Eem_ch04_0619_060343_235956",
                        help="Parent folder containing subfolders of cropped frames for each object.") # Only change these parameters
    parser.add_argument("--output_embeddings_folder", 
                        default="/xxx/Eem/Embeddings/Eem_ch04_0619_060343_235956",
                        help="Base folder to save extracted embeddings (one subfolder per object).") # Only change these parameters
    parser.add_argument("--max_workers", default=10, type=int,
                        help="Number of parallel workers to use for embedding extraction per object.") # Only change these parameters
    
    args = parser.parse_known_args()[0]
    
    results = extract_embeddings_for_all_objects(
        args.input_cropped_frames_parent,
        args.output_embeddings_folder,
        max_workers=args.max_workers
    )
    print("Extracted embeddings saved as:")
    print(results)